# ChEMBL Data Retrieval

In [113]:
import requests
from bs4 import BeautifulSoup
from chembl_webresource_client.new_client import new_client
import pandas as pd

In [2]:
molecule = new_client.molecule
drug_indication = new_client.drug_indication
target = new_client.target
activity = new_client.activity

In [21]:
# Filter out only autoimmune diseases
autoimmune_ind = drug_indication.filter(mesh_id__exact="D000224")

In [ ]:
# Exctract all MeSH terms of autoimmune diseases
# Site with MeSH terms with autoimmune diseases
url = "https://www.ncbi.nlm.nih.gov/mesh?Db=mesh&Cmd=DetailsSearch&Term=%22Autoimmune+Diseases%22%5BMeSH+Terms%5D"
response = requests.get(url)
if response.status_code == 200:
    html_content = response.text
    soup = BeautifulSoup(html_content, "html.parser")
else:
    html_content = None


#[entry['ui'] for entry in data['mesh']['descriptorRecord']]


In [ ]:
soup.find_all("span", string="Autoimmune Diseases")

[<span class="highlight" style="background-color:">Autoimmune Diseases</span>,
 <span class="highlight" style="background-color:">Autoimmune Diseases</span>,
 <span class="highlight" style="background-color:">Autoimmune Diseases</span>,
 <span class="highlight" style="background-color:">Autoimmune Diseases</span>]

In [53]:
autoimm_ul = soup.find("span", string="Autoimmune Diseases").find_all_next("ul")

In [65]:
autoimm_ul[8].find_all("a")[0].get("href")

'/mesh/68000224'

In [79]:
mesh_ids = []

for disease_a in autoimm_ul[8].find_all("a"):
    disease_url = disease_a.get('href')
    response = requests.get(f'https://www.ncbi.nlm.nih.gov{disease_url}')
    if response.status_code == 200:
        html_content = response.text
        disease_soup = BeautifulSoup(html_content, "html.parser")
        mesh_ids.append(disease_soup.find("p", string=lambda text: text and text.startswith("MeSH Unique ID:")).text.split()[-1])

In [86]:
len(mesh_ids)

50

In [ ]:
mesh_ids = [line.strip() for line in lines]
autoimmune_ind = drug_indication.filter(mesh_id__in=mesh_ids)

In [118]:
print(len(autoimmune_ind))
autoimmune_ind[0]

1256


{'drugind_id': 22607,
 'efo_id': 'EFO:0000685',
 'efo_term': 'rheumatoid arthritis',
 'indication_refs': [{'ref_id': 'NCT00048568,NCT00048581,NCT00048932,NCT00095147,NCT00122382,NCT00124449,NCT00124982,NCT00162201,NCT00162266,NCT00162279,NCT00254293,NCT00279734,NCT00279760,NCT00345748,NCT00409838,NCT00420199,NCT00484289,NCT00533897,NCT00547521,NCT00559585,NCT00663702,NCT00767325,NCT00929864,NCT00989235,NCT01000441,NCT01001832,NCT01142726,NCT01221636,NCT01295151,NCT01299961,NCT01333878,NCT01350804,NCT01351480,NCT01439204,NCT01491815,NCT01557374,NCT01602302,NCT01638715,NCT01717846,NCT01758198,NCT01844895,NCT01846975,NCT01890473,NCT02353780,NCT02466581,NCT02504268,NCT02557100,NCT02652273,NCT02722694,NCT02805010,NCT03086343,NCT03227419,NCT03414502,NCT03492658,NCT03652961,NCT03714022,NCT03737708,NCT03784261,NCT03882008,NCT04120831,NCT04255134,NCT04909801,NCT05428488,NCT05451615',
   'ref_type': 'ClinicalTrials',
   'ref_url': 'https://clinicaltrials.gov/search?term=NCT00048568%20NCT00048581

In [ ]:
indications_df = pd.DataFrame(autoimmune_ind)
chembl_ids = pd.unique(indications_df["molecule_chembl_id"])
len(chembl_ids)

In [ ]:
autoimmune_drugs = molecule.filter(molecule_chembl_id__in = chembl_ids, max_phase__gte = 3)

In [179]:
len(autoimmune_drugs)

515

In [232]:
autoimmune_biologics = autoimmune_drugs.filter(biotherapeutic__isnull=False)
autoimmune_small_mol = autoimmune_drugs.filter(biotherapeutic__isnull=True)
print(len(autoimmune_biologics))
print(len(autoimmune_small_mol))

125
390


In [ ]:
drugs_df = pd.DataFrame(autoimmune_drugs)
drugs_df[drugs_df["biotherapeutic"].notna()]["pref_name"]

88           CYCLOSPORINE
169             EXENATIDE
230            CETRORELIX
253            LANREOTIDE
256    ONABOTULINUMTOXINA
              ...        
495            BATOCLIMAB
498            FREXALIMAB
500           GEFURULIMAB
501            ZIGAKIBART
509           RILIPRUBART
Name: pref_name, Length: 125, dtype: object